In [0]:

display(dbutils.fs.ls("dbfs:/Workspace/Users/axc06310@ucmo.edu/Healthcare Analytics"))

In [0]:
%sql
SHOW CATALOGS;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.healthcare_analytics;

CREATE VOLUME IF NOT EXISTS workspace.healthcare_analytics.raw_files;

In [0]:
%sql
SELECT current_catalog() AS catalog, current_schema() AS schema;

SHOW VOLUMES IN workspace.healthcare_analytics;


In [0]:
src = "dbfs:/Workspace/Users/axc06310@ucmo.edu/Healthcare Analytics/clients.csv"

dst_base = "dbfs:/Volumes/workspace/healthcare_analytics/raw_files/"
dst = dst_base + "clients.csv"

dbutils.fs.cp(src, dst, True)
print("Copied to:", dst)

####Copy ALL CSVs from Workspace folder → Volume

In [0]:
src_base = "dbfs:/Workspace/Users/axc06310@ucmo.edu/Healthcare Analytics/"
dst_base = "dbfs:/Volumes/workspace/healthcare_analytics/raw_files/"

files = [
  "clients.csv",
  "coaches.csv",
  "coaching_sessions.csv",
  "patients.csv",
  "support_tickets.csv"
]

for f in files:
    dbutils.fs.cp(src_base + f, dst_base + f, True)

print("Copied all CSVs to Volume.")

####Create Bronze tables from the Volume files

In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS healthcare_analytics")
spark.sql("USE healthcare_analytics")

tables = {
  "clients": "clients.csv",
  "coaches": "coaches.csv",
  "coaching_sessions": "coaching_sessions.csv",
  "patients": "patients.csv",
  "support_tickets": "support_tickets.csv"
}

base_path = "dbfs:/Volumes/workspace/healthcare_analytics/raw_files/"

for name, file in tables.items():
    df = (spark.read
          .option("header", "true")
          .option("inferSchema", "true")
          .csv(base_path + file))
    
    df.write.mode("overwrite").format("delta").saveAsTable(f"bronze_{name}")

print("Bronze tables created.")

####Verifying all bronze tables are clean 
#####DESCRIBE TABLE healthcare_analytics.bronze_clients;
#####DESCRIBE TABLE healthcare_analytics.bronze_coaches;
#####DESCRIBE TABLE healthcare_analytics.bronze_coaching_sessions;
#####DESCRIBE TABLE healthcare_analytics.bronze_patients;
#####DESCRIBE TABLE healthcare_analytics.bronze_support_tickets;

In [0]:
%sql
DESCRIBE TABLE healthcare_analytics.bronze_support_tickets;


###Creating Silver Tables

In [0]:
%sql
CREATE OR REPLACE TABLE healthcare_analytics.silver_clients AS
SELECT
  client_id,
  LOWER(TRIM(client_name)) AS client_name,
  contracted_members,
  active_members,
  LOWER(TRIM(segment)) AS segment,
  go_live_date,
  LOWER(TRIM(status)) AS status
FROM healthcare_analytics.bronze_clients
WHERE client_id IS NOT NULL;

In [0]:
%sql
CREATE OR REPLACE TABLE healthcare_analytics.silver_coaches AS
SELECT
  coach_id,
  LOWER(TRIM(coach_name)) AS coach_name,
  LOWER(TRIM(region)) AS region,
  LOWER(TRIM(specialty)) AS specialty,
  LOWER(TRIM(status)) AS status
FROM healthcare_analytics.bronze_coaches
WHERE coach_id IS NOT NULL;

In [0]:
%sql
CREATE OR REPLACE TABLE healthcare_analytics.silver_patients AS
SELECT
  patient_id,
  age,
  LOWER(TRIM(gender)) AS gender,
  LOWER(TRIM(chronic_condition)) AS chronic_condition,
  LOWER(TRIM(risk_level)) AS risk_level,
  enrollment_date,
  client_id,
  LOWER(TRIM(city)) AS city,
  LOWER(TRIM(state)) AS state,
  LOWER(TRIM(member_status)) AS member_status,
  LOWER(TRIM(has_mobile_app)) AS has_mobile_app
FROM healthcare_analytics.bronze_patients
WHERE patient_id IS NOT NULL;

In [0]:
%sql
CREATE OR REPLACE TABLE healthcare_analytics.silver_coaching_sessions AS
SELECT
  session_id,
  patient_id,
  session_date,
  LOWER(TRIM(completed)) AS completed,
  coach_id,
  LOWER(TRIM(session_type)) AS session_type,
  duration_minutes,
  LOWER(TRIM(topic)) AS topic,
  LOWER(TRIM(no_show)) AS no_show
FROM healthcare_analytics.bronze_coaching_sessions
WHERE session_id IS NOT NULL;

In [0]:
%sql
CREATE OR REPLACE TABLE healthcare_analytics.silver_support_tickets AS
SELECT
  ticket_id,
  patient_id,
  created_date,
  response_time_hours,
  LOWER(TRIM(resolved)) AS resolved,
  LOWER(TRIM(issue_type)) AS issue_type,
  LOWER(TRIM(priority)) AS priority,
  LOWER(TRIM(channel)) AS channel,
  resolved_date,
  LOWER(TRIM(first_contact_resolution)) AS first_contact_resolution
FROM healthcare_analytics.bronze_support_tickets
WHERE ticket_id IS NOT NULL;

In [0]:
%sql
SHOW TABLES IN healthcare_analytics;

###Build First KPI

####KPI #1: Total Active Members

In [0]:
%sql
SELECT 
  COUNT(*) AS total_active_members
FROM healthcare_analytics.silver_patients
WHERE member_status = 'active';

####KPI #2: Total Coaching Sessions Completed

In [0]:
%sql
SELECT 
  COUNT(*) AS total_completed_sessions
FROM healthcare_analytics.silver_coaching_sessions
WHERE completed = 'yes';

####KPI #3: No-Show Rate

In [0]:
%sql
SELECT
  ROUND(
    SUM(CASE WHEN no_show = 'yes' THEN 1 ELSE 0 END) * 100.0 
    / COUNT(*), 2
  ) AS no_show_rate_percent
FROM healthcare_analytics.silver_coaching_sessions;

####KPI #4: Support Ticket Resolution Rate

In [0]:
%sql
SELECT
  ROUND(
    SUM(CASE WHEN resolved = 'yes' THEN 1 ELSE 0 END) * 100.0 
    / COUNT(*), 2
  ) AS resolution_rate_percent
FROM healthcare_analytics.silver_support_tickets;

###Gold Layer

In [0]:
%sql
CREATE OR REPLACE TABLE healthcare_analytics.gold_kpi_scorecard AS
SELECT
  current_date() AS as_of_date,

  -- KPI 1: Active members
  (SELECT COUNT(*)
   FROM healthcare_analytics.silver_patients
   WHERE member_status = 'active') AS total_active_members,

  -- KPI 2: Completed coaching sessions
  (SELECT COUNT(*)
   FROM healthcare_analytics.silver_coaching_sessions
   WHERE completed = 'yes') AS total_completed_sessions,

  -- KPI 3: No-show rate (%)
  (SELECT ROUND(
      SUM(CASE WHEN no_show = 'yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2
    )
   FROM healthcare_analytics.silver_coaching_sessions) AS no_show_rate_percent,

  -- KPI 4: Support ticket resolution rate (%)
  (SELECT ROUND(
      SUM(CASE WHEN resolved = 'yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2
    )
   FROM healthcare_analytics.silver_support_tickets) AS ticket_resolution_rate_percent;

In [0]:
%sql
SELECT * FROM healthcare_analytics.gold_kpi_scorecard;

In [0]:
%sql
CREATE OR REPLACE TABLE healthcare_analytics.gold_client_scorecard AS

WITH active_members AS (
  SELECT
    client_id,
    COUNT(*) AS active_members
  FROM healthcare_analytics.silver_patients
  WHERE member_status = 'active'
  GROUP BY client_id
),

sessions AS (
  SELECT
    p.client_id,
    COUNT(*) AS total_sessions,
    SUM(CASE WHEN s.completed = 'yes' THEN 1 ELSE 0 END) AS completed_sessions,
    SUM(CASE WHEN s.no_show = 'yes' THEN 1 ELSE 0 END) AS no_show_sessions
  FROM healthcare_analytics.silver_coaching_sessions s
  JOIN healthcare_analytics.silver_patients p
    ON s.patient_id = p.patient_id
  GROUP BY p.client_id
),

tickets AS (
  SELECT
    p.client_id,
    COUNT(*) AS total_tickets,
    SUM(CASE WHEN t.resolved = 'yes' THEN 1 ELSE 0 END) AS resolved_tickets
  FROM healthcare_analytics.silver_support_tickets t
  JOIN healthcare_analytics.silver_patients p
    ON t.patient_id = p.patient_id
  GROUP BY p.client_id
)

SELECT
  current_date() AS as_of_date,
  c.client_id,
  c.client_name,
  c.segment,
  c.status,

  COALESCE(a.active_members, 0) AS active_members,
  COALESCE(s.completed_sessions, 0) AS completed_sessions,

  -- No-show rate %
  CASE 
    WHEN COALESCE(s.total_sessions, 0) = 0 THEN 0
    ELSE ROUND(COALESCE(s.no_show_sessions, 0) * 100.0 / s.total_sessions, 2)
  END AS no_show_rate_percent,

  -- Ticket resolution rate %
  CASE 
    WHEN COALESCE(t.total_tickets, 0) = 0 THEN 0
    ELSE ROUND(COALESCE(t.resolved_tickets, 0) * 100.0 / t.total_tickets, 2)
  END AS ticket_resolution_rate_percent

FROM healthcare_analytics.silver_clients c
LEFT JOIN active_members a ON c.client_id = a.client_id
LEFT JOIN sessions s ON c.client_id = s.client_id
LEFT JOIN tickets t ON c.client_id = t.client_id;

In [0]:
%sql
SELECT * 
FROM healthcare_analytics.gold_client_scorecard
ORDER BY active_members DESC
LIMIT 20;